In [1]:
from PIL import Image
import requests
import torch
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

/home/phli/genAI/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/phli/genAI/.venv/lib64/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [2]:
# Inputs
text = ["a photo taken in the day", "a photo taken at night", "a photo of a building", "a photo taken in the urban setting"]
day_image = Image.open("/home/phli/genAI/cloudy.jpg")
night_image = Image.open("/home/phli/genAI/night.jpg")

In [3]:
input = processor(text=text, images=[day_image, night_image], return_tensors="pt", padding=True)
for key in input:
    print(key, input[key].shape)

input_ids torch.Size([4, 9])
attention_mask torch.Size([4, 9])
pixel_values torch.Size([2, 3, 224, 224])


`processor` outputs:
- `input_ids`: a tensor of shape `(batch_size, sequence_length)` containing the token ids of the input text.
- `attention_mask`: a tensor of shape `(batch_size, sequence_length)` containing the attention mask to be used in the model.
- `pixel_values`: a tensor of shape `(batch_size, num_channels, height, width)` containing the pixel values of the images. (For CLIP the image is reduced to 224x224)


In [4]:
for child in model.named_children():
    print(f"{child[0]} : {child[1].__class__.__name__}")

text_model : CLIPTextTransformer
vision_model : CLIPVisionTransformer
visual_projection : Linear
text_projection : Linear


In [7]:
outputs = model(**input)
day_image_embed, night_image_embed = outputs.image_embeds
text_embeds = outputs.text_embeds
# print(day_image_embed.shape, night_image_embed.shape, text_embeds.shape

original_logits = outputs.logits_per_image
original_probs = torch.nn.functional.softmax(original_logits, dim=1)
original_labels = torch.argmax(original_probs, dim=1)


print(text)
for i, prob in enumerate(original_probs):
    print(prob)
    print(f"Image {i}: {text[torch.argmax(prob)]}")
    print("")


print(outputs.image_embeds)
# Compute image embeddings using the submodules of the model
output_embeds2 = model.vision_model(input["pixel_values"])

# Extract the image features from the vision output
vision_outputs = model.vision_model(pixel_values=input["pixel_values"])
image_embeds = vision_outputs[1]
# Project to CLIP embedding space
image_embeds = model.visual_projection(image_embeds)
# normalized features
image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)


print(image_embeds.shape)

print(torch.allclose(outputs.image_embeds, image_embeds, atol=1e-4))

['a photo taken in the day', 'a photo taken at night', 'a photo of a building', 'a photo taken in the urban setting']
tensor([0.3544, 0.0007, 0.0203, 0.6247], grad_fn=<UnbindBackward0>)
Image 0: a photo taken in the urban setting

tensor([0.0747, 0.5184, 0.0083, 0.3986], grad_fn=<UnbindBackward0>)
Image 1: a photo taken at night

tensor([[-0.0101,  0.0575, -0.0068,  ..., -0.0039,  0.0142,  0.0042],
        [-0.0005,  0.0482,  0.0152,  ...,  0.0111,  0.0051,  0.0289]],
       grad_fn=<DivBackward0>)
torch.Size([2, 768])
True


In [6]:
# filtered_day_embed = - night_image_embed + day_image_embed
normalised_embed = day_image_embed - (night_image_embed + day_image_embed) / 2

# Calculate cosine similarity
cosine_similarity = torch.nn.functional.cosine_similarity(text_embeds, normalised_embed, dim=1)
# Scale cosine similarity by temperature
logits = cosine_similarity * model.logit_scale.exp()
probs = logits.softmax(dim=0)
print(text)
print(probs)
print(text[torch.argmax(probs)])


['a photo taken in the day', 'a photo taken at night', 'a photo of a building', 'a photo taken in the urban setting']
tensor([6.6865e-01, 8.4009e-07, 2.2401e-01, 1.0734e-01],
       grad_fn=<SoftmaxBackward0>)
a photo taken in the day
